In [1]:
import cv2

import torch
from torch import nn
from torch.utils.data import Dataset
from torchvision import models, transforms

import numpy as np

import gc


def add_zero_padding(raw_mask, raw_rgb, raw_nrg, zero_padding_dim):
    # Add zero padding
    add_height, add_width = zero_padding_dim - raw_mask.shape[0], zero_padding_dim - raw_mask.shape[1]
    padded_mask = cv2.copyMakeBorder(raw_mask, 0, add_height, 0, add_width, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    padded_rgb = cv2.copyMakeBorder(raw_rgb, 0, add_height, 0, add_width, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    padded_nrg = cv2.copyMakeBorder(raw_nrg, 0, add_height, 0, add_width, cv2.BORDER_CONSTANT, value=[0, 0, 0])

    return [padded_mask], [padded_rgb], [padded_nrg]


def split_patches(raw_mask, raw_rgb, raw_nrg, patch_dim):

    mask_list = []
    rgb_list = []
    nrg_list = []

    for used_x in range(0, int(np.ceil(raw_mask.shape[0] / patch_dim) * patch_dim), patch_dim):
        for used_y in range(0, int(np.ceil(raw_mask.shape[1] / patch_dim) * patch_dim), patch_dim):
            new_used_x = min(used_x + patch_dim, raw_mask.shape[0])
            new_used_y = min(used_y + patch_dim, raw_mask.shape[1])

            x_diff, y_diff = new_used_x - used_x, new_used_y - used_y

            mask_tile = np.zeros((patch_dim, patch_dim))
            rgb_tile = np.zeros((patch_dim, patch_dim, 3))
            nrg_tile = np.zeros((patch_dim, patch_dim, 3))

            mask_tile[:x_diff, :y_diff] = raw_mask[used_x:new_used_x, used_y:new_used_y]
            rgb_tile[:x_diff, :y_diff, :] = raw_rgb[used_x:new_used_x, used_y:new_used_y, :]
            nrg_tile[:x_diff, :y_diff, :] = raw_nrg[used_x:new_used_x, used_y:new_used_y, :]

            mask_list.append(mask_tile), rgb_list.append(rgb_tile), nrg_list.append(nrg_tile)

    return mask_list, rgb_list, nrg_list


def resize_img(raw_mask, raw_rgb, raw_nrg, fixed_size):
    mask = cv2.resize(raw_mask, (fixed_size, fixed_size), interpolation=cv2.INTER_LINEAR)
    rbg = cv2.resize(raw_rgb, (fixed_size, fixed_size), interpolation=cv2.INTER_LINEAR)
    nrg = cv2.resize(raw_nrg, (fixed_size, fixed_size), interpolation=cv2.INTER_LINEAR)

    return [mask], [rbg], [nrg]


def bootstrap_data(X, y, rotation=True, mirroring=False, return_arr=True):

    X_bootstrapped_list = []
    y_bootstrapped_list = []

    for ii in range(len(X)):

        ### No transformation
        curr_X, curr_y = X[ii], y[ii]
        X_bootstrapped_list.append(curr_X), y_bootstrapped_list.append(curr_y)

        if rotation:
            # Rotate 90 deg
            X_90, y_90 = np.rot90(curr_X).copy(), np.rot90(curr_y).copy()
            X_bootstrapped_list.append(X_90), y_bootstrapped_list.append(y_90)

            # Rotate 180 deg
            X_180, y_180 = np.rot90(X_90).copy(), np.rot90(y_90).copy()
            X_bootstrapped_list.append(X_180), y_bootstrapped_list.append(y_180)

            # Rotate 270 deg
            X_270, y_270 = np.rot90(X_180).copy(), np.rot90(y_180).copy()
            X_bootstrapped_list.append(X_270), y_bootstrapped_list.append(y_270)

        if mirroring:
            # Mirrored Horizontal Axis
            mirr_h_X, mirr_h_y = np.flip(curr_X, axis=0), np.flip(curr_y, axis=0)
            X_bootstrapped_list.append(mirr_h_X), y_bootstrapped_list.append(mirr_h_y)

            # Mirrored Vertical Axis
            mirr_v_X, mirr_v_y = np.flip(curr_X, axis=1), np.flip(curr_y, axis=1)
            X_bootstrapped_list.append(mirr_v_X), y_bootstrapped_list.append(mirr_v_y)

            # Transpose
            X_T, y_T = np.transpose(curr_X, (1, 0, 2)), np.transpose(curr_y)
            X_bootstrapped_list.append(X_T), y_bootstrapped_list.append(y_T)

            # Rotated transpose
            X_h_T, y_h_T = np.flip(X_T, axis=(0, 1)), np.flip(y_T)
            X_bootstrapped_list.append(X_h_T), y_bootstrapped_list.append(y_h_T)
        
    if return_arr:
        X_bootstrapped_list = np.stack(X_bootstrapped_list)
        y_bootstrapped_list = np.stack(y_bootstrapped_list)

    return X_bootstrapped_list, y_bootstrapped_list


def calc_iou(pred, y_true, threshholds):

    flattened_pred = np.concat([x.flatten() for x in pred])
    flattened_true = np.concat([x.flatten() for x in y_true])
    iou_list = []

    for cutoff in threshholds:
        pred_class = flattened_pred > cutoff
        iou_list.append(np.logical_and(pred_class, flattened_true).sum() / np.logical_or(pred_class, flattened_true).sum())

    return iou_list


def get_train_iou_list(model, device, train_loader, val_loader, masks_list, resized_masks_list=[], use_resize=True):
    train_preds_list = []
    train_idx_list = []

    model.eval()
    with torch.no_grad():
        for ii, (X, y, indices) in enumerate(train_loader):
            X, y = X.to(device), y.to(device)
            train_preds_list.append(model(X).reshape(y.shape).cpu().detach().numpy())
            train_idx_list.extend([(indices[0][ii].item(), indices[1][ii].item()) for ii in range(len(indices[0]))])

        # Include validation set in cutoff selection
        for ii, (X, y, indices) in enumerate(val_loader):
            X, y = X.to(device), y.to(device)
            train_preds_list.append(model(X).reshape(y.shape).cpu().detach().numpy())
            train_idx_list.extend([(indices[0][ii].item(), indices[1][ii].item()) for ii in range(len(indices[0]))])

    torch.cuda.empty_cache()
    gc.collect()

    resized_train_pred_list = []
    train_preds = np.concat(train_preds_list, axis=0)
    filtered_train_preds = [x for x, (_, non_transformed) in zip(train_preds, train_idx_list) if non_transformed]
    train_true_list = [masks_list[idx] / 255.0 for idx, non_transformed in train_idx_list if non_transformed]

    threshholds = np.arange(0, 1.02, 0.02)
    smoothed_train_preds = [torch.sigmoid(torch.tensor(x)).numpy() for x in filtered_train_preds]
    train_iou_list = calc_iou(smoothed_train_preds, train_true_list, threshholds)

    resized_train_iou_list = []

    if use_resize:
        for ii in range(len(train_idx_list)):
            if train_idx_list[ii][1]:
                idx = train_idx_list[ii][0]
                shape = resized_masks_list[idx].shape
                resized_pred = cv2.resize(train_preds[ii], shape[-1::-1], interpolation=cv2.INTER_LINEAR)
                resized_train_pred_list.append(resized_pred)

        resized_train_true_list = [resized_masks_list[idx] / 255.0 for idx, non_transformed in train_idx_list if non_transformed]
        smoothed_resized_train_pred_list = [torch.sigmoid(torch.tensor(x)).numpy() for x in resized_train_pred_list]
        resized_train_iou_list = calc_iou(smoothed_resized_train_pred_list, resized_train_true_list, threshholds)

    return train_iou_list, threshholds, resized_train_iou_list


def get_test_iou_list(model, device, test_loader, opt_cutoff, masks_list, resized_masks_list, use_resize):

    test_preds_list = []
    test_idx_list = []

    with torch.no_grad():
        for X, y, indices in test_loader:
            X, y = X.to(device), y.to(device)
            preds = model(X).reshape(y.shape).cpu().detach().numpy()
            test_preds_list.append(preds)
            test_idx_list.extend(indices)


    torch.cuda.empty_cache()
    gc.collect()

    resized_test_pred_list = []
    test_preds = np.concat(test_preds_list, axis=0)
    smoothed_test_preds = [torch.sigmoid(torch.tensor(x)).numpy() for x in test_preds]

    test_true_list = [masks_list[idx] / 255.0 for idx in test_idx_list]
    threshholds = np.arange(0, 1.02, 0.02)
    test_iou_list = calc_iou(smoothed_test_preds, test_true_list, [opt_cutoff])[0]
    resized_test_iou_list = []

    if use_resize:
        for ii in range(len(test_idx_list)):
            idx = test_idx_list[ii]
            shape = resized_masks_list[idx].shape
            resized_pred = cv2.resize(test_preds[ii], shape[-1::-1], interpolation=cv2.INTER_LINEAR)
            resized_test_pred_list.append(resized_pred)

        resized_train_true_list = [resized_masks_list[idx] / 255.0 for idx in test_idx_list]
        smoothed_resized_test_pred_list = [torch.sigmoid(torch.tensor(x)).numpy() for x in resized_test_pred_list]
        resized_test_iou_list = calc_iou(smoothed_resized_test_pred_list, resized_train_true_list, [opt_cutoff])[0]

    return test_iou_list, threshholds, resized_test_iou_list


def check_stopping_tol(val_loss_list, patience, tol_neg_sig):
    last = round(val_loss_list[-(patience + 1)] - val_loss_list[-1], tol_neg_sig) <= (0.1) ** (tol_neg_sig)
    second_last = round(val_loss_list[-(patience + 2)] - val_loss_list[-2], tol_neg_sig) <= (0.1) ** (tol_neg_sig)

    return last and second_last


class SquareDataset(Dataset):
    def __init__(self, data, mask, id):
        # Shift chanels to second dimension
        self.data = torch.tensor(data).permute(0, 3, 1, 2)
        self.labels = torch.tensor(mask)
        self.id = id

    def __getitem__(self, idx):
        return self.data[idx, :, :], self.labels[idx, :, :], self.id[idx]

    def __len__(self):
        return len(self.data)
    
    
class VariableDataset(Dataset):
    def __init__(self, data, mask):
        self.data = [torch.tensor(x).permute(2, 0, 1) for x in data]
        self.labels = [torch.tensor(x) for x in mask]

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

    def __len__(self):
        return len(self.data)


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, use_backbone=False, use_sigmoid=False):
        super(UNet, self).__init__()
        self.use_backbone = use_backbone
        self.use_sigmoid = use_sigmoid

        # self.use_backone = use_backbone

        # if use_backbone:
        #     backbone = models.convnext_large(weights='DEFAULT')

        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.down4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(64, out_channels, 1)


    def _vanilla_forward(self, x):
        c1 = self.down1(x)
        p1 = self.pool1(c1)
        c2 = self.down2(p1)
        p2 = self.pool2(c2)
        c3 = self.down3(p2)
        p3 = self.pool3(c3)
        c4 = self.down4(p3)
        p4 = self.pool4(c4)

        bn = self.bottleneck(p4)

        u4 = self.up4(bn)
        d4 = self.dec4(torch.cat([u4, c4], dim=1))
        u3 = self.up3(d4)
        d3 = self.dec3(torch.cat([u3, c3], dim=1))
        u2 = self.up2(d3)
        d2 = self.dec2(torch.cat([u2, c2], dim=1))
        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat([u1, c1], dim=1))

        return d1
    

    def _backbone_encoder_forward(self, x):
        return NotImplementedError


    def forward(self, x):

        d1 = self._vanilla_forward(x) if not self.use_backbone else self._backbone_forward(x)
        out = self.final(d1)

        if self.use_sigmoid:
            out = torch.sigmoid(out)

        return out
    



# Feature extraction using resnet
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet_model = nn.Sequential(*list(models.resnet50(weights='ResNet50_Weights.DEFAULT').children())[:-2])

    def forward(self, input_tensor):
        features = self.resnet_model(input_tensor)  
        return features
    

# Converting previous functions to class type nn for modelling
# Attention Module
# we will create an attention module which will highlight areas of interest.
# we need to create a query key and value for the attention module. We will use the softmax activation function 
# and linear functions for query, key and value. (From my understanding of the paper, in the methodology section.)

class AttentionModule(nn.Module):
    def __init__(self):
        super().__init__()
        # Declaring convolution layers for Query Key and Value
        self.Q = nn.Conv2d(2048, 2048, kernel_size=1)
        self.K = nn.Conv2d(2048, 2048, kernel_size=1)
        self.V = nn.Conv2d(2048, 2048, kernel_size=1)

    def forward(self, Feature_extracted_tensor):
        # Declare Q , K and V
        Q = self.Q(Feature_extracted_tensor)
        K = self.K(Feature_extracted_tensor)
        V = self.V(Feature_extracted_tensor)
                
        # Calculate attention score based on the formula in the literature
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / (Feature_extracted_tensor.size(-1) ** 0.5)
        # Apply softmax to get attention weights
        attention_weights = nn.Softmax(dim=-1)(attention_scores)
        # Apply attention weights to values
        features_attention_adjusted = torch.matmul(attention_weights, V)
        return features_attention_adjusted, attention_weights
    

# Segment Decoder to piece features back together to compare with ground truth mask
class SegmentDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.upsize_1 = nn.Sequential(
            nn.ConvTranspose2d(2048, 256, kernel_size=2, stride=2),  # 7x7 -> 14x14
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )
        self.upsize_2 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),  # 14x14 -> 28x28
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )
        self.upsize_3 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),  # 28x28 -> 56x56
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.upsize_4 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),  # 56x56 -> 112x112
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.upsize_5 = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2),  # 112x112 -> 224x224
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )

        # Final 1x1 conv to get per-pixel class scores 
        self.final_conv = nn.Conv2d(16, 1, kernel_size=1)
    def forward(self, input_tensor):
        # upscale extracted features back to 224 * 224 and segment it to binary white and black.
        upsize_1 = self.upsize_1(input_tensor)  # 7x7 → 14x14
        upsize_2 = self.upsize_2(upsize_1)  # → 28x28
        upsize_3 = self.upsize_3(upsize_2)  # → 56x56
        upsize_4 = self.upsize_4(upsize_3)  # → 112x112
        upsize_5 = self.upsize_5(upsize_4)  # → 224x224
        final = self.final_conv(upsize_5)
        return final


class ADA_single_domain(nn.Module):
    def __init__(self):
        super(ADA_single_domain, self).__init__()
        self.feature_extractor = FeatureExtractor()
        self.attention_module =  AttentionModule()
        self.segment_decoder = SegmentDecoder()

    # function to convert image to tensor to feed into feature extractor
    # @staticmethod
    def _image_converted_to_tensor(image):
        # normalized_image = cv2.normalize(image, None, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_32F)
        image_transformed = (image).requires_grad_()
        image_batched = image_transformed.unsqueeze(0)
        return image_batched

    def forward(self, images):
        # image_converted_tensor = ADA_single_domain._image_converted_to_tensor(images)
        images = images.requires_grad_()
        features_extracted = self.feature_extractor(images)
        features_attention_adjusted, attention_weights = self.attention_module(features_extracted)
        output = self.segment_decoder(features_attention_adjusted)

        return output

# transforms.ToTensor()
# resnet = models.resnet50(weights='DEFAULT')
# resnet_model = nn.Sequential(*list(resnet.children())[:-2])
# feature_extractor = FeatureExtractor(resnet_model)


